## Project: Labels vs. Scale: Breaking Even on Banking77 with Encoders and Decoders

#### Frontier model zero-shot control

This notebook runs **Claude Fable 5.2** zero-shot on the validation (385) and test (3,080) sets, with the **same prompt and the same matching rule as Qwen zero-shot**. It writes two prediction files in the shared format (`frontier_zeroshot_seed0_val.csv` and `frontier_zeroshot_seed0_test.csv`) into `results/`.

Run order: Setup, Smoke test (5 examples), Full validation run, Full test run. Progress is cached on disk, so a stopped run resumes where it left off.

#### Setup

In [ ]:
# Install the Anthropic SDK (skip if already installed)
!pip install -q anthropic

In [ ]:
import os, sys, json, time, getpass, datetime
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import anthropic

# Run from the repo root so the relative paths below work
# os.chdir(os.path.expanduser("~/github/DATASCI266-Banking77-Encoder-vs-Decoder"))
sys.path.insert(0, "scripts")
from predictions import match_intent, write_predictions
import evaluation as ev

# Register the frontier model in the loaded module (also add it to scripts/evaluation.py on disk)
ev.MODEL_TYPE["frontier"] = "decoder"
print("Registered models:", [m for m in ev.MODEL_TYPE if "frontier" in m or "qwen" in m])

#### API key and settings

The key is read from the `ANTHROPIC_API_KEY` environment variable. If it is missing, would be prompted for it. The key is never written to a file.

In [ ]:
# Read the API key from the environment, or ask for it once
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

# Run settings (record these in the report's Methods section)
MODEL_ID = "claude-fable-5-2"     # API model string
MODEL_NAME = "frontier"           # Name used in the prediction files
BUDGET, SEED = "zeroshot", 0      # Zero-shot runs use seed 0 only
MAX_TOKENS = 40                   # Intent names are short
TEMPERATURE = 0                   # Deterministic as far as the API allows
WORKERS = 4                       # Parallel requests; lower this if rate limits are hit
CACHE_DIR = "reports/frontier_cache"   # Not in results/, so the scorer doesn't sees it
os.makedirs(CACHE_DIR, exist_ok=True)

# Prices in USD per million tokens (Anthropic standard rates)
PRICE_IN, PRICE_OUT = 8.0, 40.0

client = anthropic.Anthropic(max_retries=6)   # The SDK retries rate limits and transient errors

#### Data and prompt

The system line, the intent list and the message template are copied from the Qwen notebook (`Final_Decoder_Finetune.ipynb`), so Qwen zero-shot and the frontier model see exactly the same text.

In [ ]:
# Load the label names, the validation set and the test set
labels = json.load(open("data/raw/label_names.json"))
val = pd.read_parquet("data/splits/val.parquet")
test = pd.read_parquet("data/raw/test.parquet")
print("Val:", val.shape, "| Test:", test.shape, "| Intents:", len(labels))

# Same prompt as the Qwen runs
SYSTEM = ("You classify banking customer messages. Reply with exactly one intent "
          "name from the list, copied exactly, and nothing else.")
INTENT_LIST = "\n".join(labels)

def user_text(message):
    return f"Intents:\n{INTENT_LIST}\n\nMessage: {message}\n\nIntent:"

print(user_text(val["text"].iloc[0])[-200:])

#### Classification helpers

Each query is one API call. Results are cached per example in a JSON-lines file, so a run can be stopped and resumed. Token counts are kept for the cost estimate.

In [ ]:
USED_NO_TEMPERATURE = False   # Set to True if the API rejects the temperature setting

def call_model(message):
    """Send one query and return (raw_text, input_tokens, output_tokens, model_returned)."""
    global USED_NO_TEMPERATURE
    kwargs = dict(model=MODEL_ID, max_tokens=MAX_TOKENS, system=SYSTEM,
                  messages=[{"role": "user", "content": user_text(message)}])
    if not USED_NO_TEMPERATURE:
        kwargs["temperature"] = TEMPERATURE
    try:
        resp = client.messages.create(**kwargs)
    except anthropic.BadRequestError as e:
        # Some models reject sampling settings; retry once without temperature and note it
        if "temperature" in str(e).lower() and "temperature" in kwargs:
            USED_NO_TEMPERATURE = True
            print("Note: temperature was rejected, running with the API default instead")
            kwargs.pop("temperature", None)
            resp = client.messages.create(**kwargs)
        else:
            raise
    text = "".join(b.text for b in resp.content if b.type == "text").strip()
    return text, resp.usage.input_tokens, resp.usage.output_tokens, resp.model

def classify(df, split, limit=None):
    """Classify every row of df (or the first `limit` rows), using and filling the cache."""
    path = f"{CACHE_DIR}/{split}.jsonl"
    cache = {}
    if os.path.exists(path):
        for line in open(path):
            r = json.loads(line)
            cache[r["example_id"]] = r
    ids = list(range(len(df) if limit is None else limit))
    todo = [i for i in ids if i not in cache]
    print(f"{split.capitalize()}: {len(ids) - len(todo)} cached, {len(todo)} to run")

    def work(i):
        text, tin, tout, mdl = call_model(df["text"].iloc[i])
        return {"example_id": i, "raw": text, "in_tok": tin, "out_tok": tout, "model": mdl}

    start = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool, open(path, "a") as fh:
        for n, r in enumerate(pool.map(work, todo), 1):
            cache[r["example_id"]] = r
            fh.write(json.dumps(r) + "\n"); fh.flush()
            if n % 100 == 0:
                print(f"Progress: {n}/{len(todo)} done, {time.time() - start:.0f} s")
    return [cache[i] for i in ids]

def summarize(rows, df, split):
    """Match the raw outputs to intents and print accuracy, unmatched rate and cost."""
    raw = [r["raw"] for r in rows]
    pred = np.array([match_intent(t) for t in raw])
    y = df["label"].to_numpy()[:len(rows)]
    tin, tout = sum(r["in_tok"] for r in rows), sum(r["out_tok"] for r in rows)
    cost = tin / 1e6 * PRICE_IN + tout / 1e6 * PRICE_OUT
    print(f"{split}: Accuracy={(pred == y).mean():.3f} | Unmatched={(pred == -1).mean():.3f} | "
          f"Input tokens={tin:,} | Output tokens={tout:,} | Cost=${cost:.2f}")
    return raw, pred, tin, tout

#### Smoke test

Five validation examples. Check that the raw outputs are clean intent names and that the returned model string is the one we expect. Stop here if anything looks wrong.

In [ ]:
# Use a separate cache name so smoke-test rows never mix with the real validation cache
smoke = []
for i in range(5):
    text, tin, tout, mdl = call_model(val["text"].iloc[i])
    smoke.append((val["text"].iloc[i], text, labels[val["label"].iloc[i]], match_intent(text), mdl))
for msg, text, gold, p, mdl in smoke:
    print(f"Message: {msg}\n  Model output: {text!r} | Gold: {gold} | Matched index: {p}\n")
print("Model returned:", smoke[0][4])

#### Full validation run (385 examples)

In [ ]:
val_rows = classify(val, "val")
val_raw, val_pred, val_tin, val_tout = summarize(val_rows, val, "Val")
write_predictions(MODEL_NAME, BUDGET, SEED, "val", val_pred, raw_outputs=val_raw)

#### Full test run (3,080 examples)



In [ ]:
test_rows = classify(test, "test")
test_raw, test_pred, test_tin, test_tout = summarize(test_rows, test, "Test")
write_predictions(MODEL_NAME, BUDGET, SEED, "test", test_pred, raw_outputs=test_raw)

#### Save the run details

In [ ]:
info = {
    "model_requested": MODEL_ID,
    "models_returned": sorted({r["model"] for r in val_rows + test_rows}),
    "date_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
    "temperature": None if USED_NO_TEMPERATURE else TEMPERATURE,
    "max_tokens": MAX_TOKENS,
    "input_tokens": val_tin + test_tin,
    "output_tokens": val_tout + test_tout,
    "cost_usd": round((val_tin + test_tin) / 1e6 * PRICE_IN + (val_tout + test_tout) / 1e6 * PRICE_OUT, 2),
    "val_unmatched": int((val_pred == -1).sum()),
    "test_unmatched": int((test_pred == -1).sum()),
}
os.makedirs("reports", exist_ok=True)
json.dump(info, open("reports/frontier_run_info.json", "w"), indent=2)
print(json.dumps(info, indent=2))